<a href="https://colab.research.google.com/github/psept77-star/CAPA_Effectiveness_Prediction/blob/main/CAPA_Effectiveness_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import numpy as np
import pandas as pd
import xgboost as xgb
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, f1_score


In [26]:
# STEP 1: HISTORICAL DATA LOADING & STANDARDIZATION

df = pd.read_excel('/content/sample_data/CAPA_Updated_Early_Stage_Dataset_4000.xlsx')

# Standardize case and strip hidden spaces for ALL categorical text to eliminate case-sensitivity
for col in ["Department", "Severity", "Overdue", "Recurrence_90d"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

# Mocking placeholder fields if they don't exist in historical dataset yet
if "Remedial_Action" not in df.columns: df["Remedial_Action"] = "Immediate containment applied"
if "CAPA_Action" not in df.columns: df["CAPA_Action"] = "Update standard operating procedures and retrain staff"
if "Possible_Root_Cause" not in df.columns: df["Possible_Root_Cause"] = df["Root_Cause"]

# Standardize case for the new investigation field
df["Possible_Root_Cause"] = df["Possible_Root_Cause"].astype(str).str.strip().str.title()

# Natively encode standardized categories
dept_encoder = LabelEncoder().fit(df["Department"])
sev_encoder = LabelEncoder().fit(df["Severity"])
possible_rc_encoder = LabelEncoder().fit(df["Possible_Root_Cause"])
overdue_encoder = LabelEncoder().fit(df["Overdue"])
recurrence_encoder = LabelEncoder().fit(df["Recurrence_90d"])

df["Dept_encoded"] = dept_encoder.transform(df["Department"])
df["Sev_encoded"] = sev_encoder.transform(df["Severity"])
df["RC_encoded"] = possible_rc_encoder.transform(df["Possible_Root_Cause"])
df["Overdue_encoded"] = overdue_encoder.transform(df["Overdue"])
df["Recurrence_encoded"] = recurrence_encoder.transform(df["Recurrence_90d"])

# display data
display(df.head())

,CAPA_ID,Department,Severity,Possible_Root_Cause,Remedial_Action,CAPA_Action,Closure_Time_Days,Overdue,Recurrence_90d,CAPA_Effective,Dept_encoded,Sev_encoded,RC_encoded,Overdue_encoded,Recurrence_encoded
0,CAPA-0001,Regulatory Affairs,Major,Software Bug,Restarted system module and reverted to stable...,Deploy hotfix release v2.4.1 and implement reg...,65,Yes,No,1,5,1,3,1,0
1,CAPA-0002,Packaging,Major,Human Error,Retrained the immediate operator and reinforce...,Implement structural safety switches and secon...,111,Yes,Yes,1,2,1,1,1,1
2,CAPA-0003,Packaging,Critical,Human Error,Retrained the immediate operator and reinforce...,Implement structural safety switches and secon...,154,Yes,No,1,2,0,1,1,0
3,CAPA-0004,Engineering,Minor,Human Error,Retrained the immediate operator and reinforce...,Implement structural safety switches and secon...,47,Yes,Yes,1,0,2,1,1,1
4,CAPA-0005,Regulatory Affairs,Minor,Human Error,Retrained the immediate operator and reinforce...,Implement structural safety switches and secon...,91,Yes,No,1,5,2,1,1,0


In [27]:
# STEP 2: FEATURE ENGINEERING & DATA SPLITTING

# Vectorize action plan text fields using TF-IDF
tfidf_remedial = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_capa = TfidfVectorizer(max_features=100, stop_words='english')

X_text_remedial = tfidf_remedial.fit_transform(df["Remedial_Action"]).toarray()
X_text_capa = tfidf_capa.fit_transform(df["CAPA_Action"]).toarray()

# Combine structured fields + action text arrays
X_categorical = df[["Dept_encoded", "Sev_encoded", "RC_encoded", "Overdue_encoded"]].values
X_combined = np.hstack((X_categorical, X_text_remedial, X_text_capa))

# Targets
y_clf = df[["CAPA_Effective", "Recurrence_encoded"]]
y_reg = df["Closure_Time_Days"]

# Split data for evaluation
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_combined, y_clf, test_size=0.2, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_combined, y_reg, test_size=0.2, random_state=42)



In [28]:
# STEP 3: EXPLICIT MODEL SELECTION STEP (BENCHMARKING)

print("--- Evaluating Models ---")

# 1. Define the 4 models to test
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    "Random Forest": RandomForestClassifier(random_state=42)
}

scores = {}
trained_models = {}

# 2. Train and score each model automatically
for name, model in models.items():
    # Wrap in MultiOutputClassifier to handle both prediction targets at once
    wrapped_model = MultiOutputClassifier(model).fit(X_train_c, y_train_c)
    preds = wrapped_model.predict(X_test_c)

    # Calculate average accuracy across both targets
    acc_effective = accuracy_score(y_test_c["CAPA_Effective"], preds[:, 0])
    acc_recurrence = accuracy_score(y_test_c["Recurrence_encoded"], preds[:, 1])

    scores[name] = (acc_effective + acc_recurrence) / 2
    trained_models[name] = wrapped_model
    print(f" -> {name}: {scores[name]:.2%}")

# 3. Automatically select the one with the highest overall score
best_model_name = max(scores, key=scores.get)
clf_model = trained_models[best_model_name]

print(f"\n Chosen Model: {best_model_name}")

# Keep the timeline regressor consistent
reg_model = xgb.XGBRegressor(random_state=42).fit(X_train_r, y_train_r)


--- Evaluating Models ---
 -> Logistic Regression: 49.69%
 -> SVM: 50.19%
 -> XGBoost: 50.69%
 -> Random Forest: 50.19%

 Chosen Model: XGBoost


In [29]:
# STEP 4: CONVERGED LIVE INFERENCE PIPELINE

def analyze_early_stage_capa(department, severity, possible_root_cause, remedial_text, capa_text, is_overdue):
    try:
        dept_clean = str(department).strip().title()
        sev_clean = str(severity).strip().title()
        rc_clean = str(possible_root_cause).strip().title()
        overdue_clean = str(is_overdue).strip().title()

        if rc_clean not in possible_rc_encoder.classes_:
            print(f"❌ Error: '{possible_root_cause}' is not a recognized Root Cause.")
            print(f"Valid entries are: {list(possible_rc_encoder.classes_)}")
            return

        # Encode inputs natively
        encoded_dept = dept_encoder.transform([dept_clean])[0]
        encoded_sev = sev_encoder.transform([sev_clean])[0]
        encoded_rc = possible_rc_encoder.transform([rc_clean])[0]
        encoded_overdue = overdue_encoder.transform([overdue_clean])[0]

        # Vectorize text features
        vec_remedial = tfidf_remedial.transform([remedial_text]).toarray()
        vec_capa = tfidf_capa.transform([capa_text]).toarray()

        # Combine into standardized feature template array
        vec_categorical = np.array([[encoded_dept, encoded_sev, encoded_rc, encoded_overdue]])
        model_input = np.hstack((vec_categorical, vec_remedial, vec_capa))

        # Run deployed model structures
        clf_preds = clf_model.predict(model_input)[0]
        clf_probs = clf_model.predict_proba(model_input)

        pred_effective = clf_preds[0]
        pred_recurrence = clf_preds[1]
        predicted_days = reg_model.predict(model_input)[0]

        # Extract classification confidences cleanly from structured wrapper lists
        eff_confidence = clf_probs[0][0][pred_effective] * 100
        rec_confidence = clf_probs[1][0][pred_recurrence] * 100

        eff_label = "✅ EFFECTIVE" if pred_effective == 1 else "🔴 INEFFECTIVE (High Rework Risk)"
        rec_label = "⚠️ HIGH RISK" if recurrence_encoder.classes_[pred_recurrence] == "Yes" else "✅ LOW RISK"

        print("==================================================")
        print("      EARLY STAGE QUALITY INTELLIGENCE ENGINE     ")
        print("==================================================")
        print(f" DRAFT STAGE PROFILE: {sev_clean} | {dept_clean}")
        print(f" Validated Root Cause: {rc_clean}")
        print("--------------------------------------------------")
        print(f" 1. ACTION PLAN VIABILITY:")
        print(f"   -> {eff_label} ({eff_confidence:.1f}% Confidence) [Engine: {best_model_name}]")
        print(f" 2. 90-DAY DEVIATION RECURRENCE RISK:")
        print(f"   -> {rec_label} ({rec_confidence:.1f}% Confidence)")
        print(f" 3. FORECASTED CYCLE TIME:")
        print(f"   -> Estimated {max(0, int(predicted_days))} days to complete verification")
        print("==================================================\n")

    except Exception as e:
        print(f"❌ Processing error encountered: {e}")



In [30]:
# STEP 5: INTERACTIVE UI LAYOUT (IPYWIDGETS)

dept_dropdown = widgets.Dropdown(options=list(dept_encoder.classes_), description='Department:')
sev_dropdown = widgets.Dropdown(options=list(sev_encoder.classes_), description='Severity:')
overdue_dropdown = widgets.Dropdown(options=['No', 'Yes'], description='Overdue Status:')
rc_text_input = widgets.Text(placeholder='Type cause (e.g. human error, software bug)...', description='Root Cause:')

remedial_input = widgets.Textarea(placeholder='What immediate actions were taken?', description='Remedial:')
capa_input = widgets.Textarea(placeholder='Describe the planned preventive/corrective actions...', description='CAPA Plan:')

predict_button = widgets.Button(description='🔮 Run Early Assessment', button_style='primary')
output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output()  # Clears the previous prediction output before running a new one
        analyze_early_stage_capa(
            department=dept_dropdown.value,
            severity=sev_dropdown.value,
            possible_root_cause=rc_text_input.value,
            remedial_text=remedial_input.value,
            capa_text=capa_input.value,
            is_overdue=overdue_dropdown.value
        )

# Link the button click to the execution function
predict_button.on_click(on_predict_clicked)

# Assemble everything into a clean vertical form layout
input_form = widgets.VBox([
    widgets.HTML("<h3>🧠 Phase-1 CAPA Risk & Viability Estimator</h3>"),
    dept_dropdown, sev_dropdown, rc_text_input, overdue_dropdown,
    remedial_input, capa_input, widgets.HTML("<br>"), predict_button
])

# Render the application form interface inside your notebook cell
display(input_form, output_area)


Output()